## About this Entry

In the previous few entries, I have illustrated some features of one of the most important tools in a data analyst's toolbox: SQL. Not everything can be done in SQL, though, which means that having other tools is indispensable.

This entry is the first one in a series that will talk about another crucial piece of technology — my favorite one, to be honest — for dealing with data: Python.

At the start of their journey, data analysts must choose a technology in which to specialize in order to handle data in ways that SQL cannot or is not optimized to do. Because many companies don't have their data properly structured in relational databases and use spreadsheets instead, one good option is to become an Excel wizard, learning how to leverage VLookups and Macros to solve daily data needs — I'll cover Excel data analysis in the future. Another, more advanced option is to learn R, a programming language for statistical computing and data visualization specially popular among statisticians and academics. Yet another option, very popular in big enterprises, is to resort to business intelligence tools like Power BI and Tableau.


My choice was to specialize in Python, an open-source, general-purpose programming language capable of doing much more than "just" data analysis, but nonetheless exquisitely prepared to handle this kind of work on account of a rich set of libraries dedicated to that. In reality, every data analyst will benefit from knowing how to navigate all of the technologies above. However, much like with our native tongue, there is always that one with which we feel more at ease. 


As I said, not all companies have the capacity to properly structure their data in relational databases — some, on the other hand, have to deal with even more complex data that can't be handled by them. Either way, Python is greatly equipped for dealing with data stored in numerous formats. The code I am uploading to my portfolio today shows just how easy it is to use [Pandas](https://pandas.pydata.org/) (one of Python's libraries dedicated to data analysis and manipulation) to ingest data from multiple file formats and save it in a DataFrame (Pandas' data structure for handling tabular data). In a situation where a company's employee data is spread across multiple files with different formats (CSV, Excel, and JSON), we are easily able to merge them in a single structure and save it in any format we want (or even feed it to a database using SQL).

## Analysis Context

In this fictional situation, we are working for a company that keeps their data spread across multiple files in different formats. We want to group said data in one single structure so we can not only work more easily with it but also improve its governance, that is, assure its quality, security, availability, and up-to-dateness.

In [23]:
import pandas as pd
pd.set_option("display.max_columns", None)

# Dataframes
office_addresses = pd.read_csv("datasets/office_addresses.csv")

employee_addresses = pd.read_excel("datasets/employee_information.xlsx", sheet_name=0)

emergency_contacts = pd.read_excel("datasets/employee_information.xlsx", sheet_name="emergency_contacts")
emergency_contacts.columns = ["employee_id", "last_name", "first_name","emergency_contact",
                              "emergency_contact_number", "relationship"]

employee_roles = pd.read_json("datasets/employee_roles.json", orient="index")

In [3]:
office_addresses

,office,office_country,office_city,office_street,office_street_number
0,Leuven Office,BE,Leuven,Martelarenlaan,38
1,ESB Office,US,New York City,Fifth Avenue,350
2,WeWork Office,GB,London,Old Street,207


In [24]:
employee_addresses

,employee_id,employee_last_name,employee_first_name,employee_country,employee_city,employee_street,employee_street_number
0,N7PFTN,Smith,Grace,US,New-York,Perry Street,75
1,ZOXBQ4,Davis,Alice,BE,Leuven,Bondgenotenlaan,270
2,5224BT,Williams,Alice,GE,Berlin,Friedrichstraße,425
3,O53LUV,Wilson,Eve,US,New-York,Madison Avenue,335
4,SUEX6P,Johnson,David,GB,London,Baker Street,105
...,...,...,...,...,...,...,...
495,3Q85AT,Davis,Grace,US,New-York,5th Avenue,180
496,LGT30D,Johnson,Ivy,FR,Paris,Rue de l'Université,414
497,F7666L,Johnson,Eve,FR,Lyon,Boulevard Haussmann,169
498,E4YN8M,Smith,Alice,US,New-York,5th Avenue,382


In [25]:
emergency_contacts

,employee_id,last_name,first_name,emergency_contact,emergency_contact_number,relationship
0,N7PFTN,Smith,Grace,Bob Davis,+12-628-1533-23,Sister
1,ZOXBQ4,Davis,Alice,Eve Smith,+72-654-9963-83,Mother
2,5224BT,Williams,Alice,Hank Smith,+16-280-4352-30,Father
3,O53LUV,Wilson,Eve,John Jones,+31-604-2134-39,Mother
4,SUEX6P,Johnson,David,Bob Brown,+35-753-5910-79,Sister
...,...,...,...,...,...,...
495,3Q85AT,Davis,Grace,Carol Wilson,+73-147-3484-43,Friend
496,LGT30D,Johnson,Ivy,Frank Brown,+32-656-3523-31,Mother
497,F7666L,Johnson,Eve,Hank Smith,+4-338-4828-70,Friend
498,E4YN8M,Smith,Alice,Carol Miller,+47-108-8722-27,Wife


In [27]:
employee_roles.reset_index(inplace=True, names='employee_id')
employee_roles

,employee_id,title,monthly_salary,team
0,N7PFTN,HR Specialist,$2500,People Operations
1,ZOXBQ4,UX Designer,$3100,Design
2,5224BT,Office Manager,$2000,People Operations
3,O53LUV,Customer Support,$1800,Support
4,SUEX6P,Data Analyst,$3200,Data
...,...,...,...,...
495,3Q85AT,UX Designer,$3100,Design
496,LGT30D,Product Manager,$3700,Product
497,F7666L,CEO,$4500,Leadership
498,E4YN8M,UX Designer,$3100,Design


Please note that the values used for refering to the city of New York are inconsistent between the tables "office_addresses" and "employee_addresses". This is a common issue when data is stored in multiple sources and, if we don't fix it, will lead to loss of data when building our final (merged) table, since Pandas will not find a match between "New-York" and "New York City". We can elegantly solve problems like that using fuzzy string matching — or [approximate string matching](https://en.wikipedia.org/wiki/Approximate_string_matching), in more technical terms. However, in this context, that would be using a complex solution to a simple problem. Instead, we can simply replace the values in each dataframe:

In [28]:
office_addresses.replace("New York City", "New York", inplace=True)
employee_addresses.replace("New-York", "New York", inplace=True)

Now we can merge the tables.

In [31]:
#Final df
employees_final = employee_addresses.merge(emergency_contacts, on='employee_id')\
                                    .merge(employee_roles, on='employee_id')\
                                    .merge(office_addresses, left_on='employee_city', right_on='office_city', how="left")\
                                    .fillna("Remote")\
                                    .set_index('employee_id')
employees_final = employees_final[["employee_first_name", "employee_last_name", "employee_country",
                                   "employee_city", "employee_street", "employee_street_number",
                                   "emergency_contact", "emergency_contact_number", "relationship",
                                   "monthly_salary", "team", "title", "office", "office_country",
                                   "office_city", "office_street", "office_street_number"]]

employees_final

,employee_first_name,employee_last_name,employee_country,employee_city,employee_street,employee_street_number,emergency_contact,emergency_contact_number,relationship,monthly_salary,team,title,office,office_country,office_city,office_street,office_street_number
employee_id,,,,,,,,,,,,,,,,,
N7PFTN,Grace,Smith,US,New York,Perry Street,75,Bob Davis,+12-628-1533-23,Sister,$2500,People Operations,HR Specialist,ESB Office,US,New York,Fifth Avenue,350.0
ZOXBQ4,Alice,Davis,BE,Leuven,Bondgenotenlaan,270,Eve Smith,+72-654-9963-83,Mother,$3100,Design,UX Designer,Leuven Office,BE,Leuven,Martelarenlaan,38.0
5224BT,Alice,Williams,GE,Berlin,Friedrichstraße,425,Hank Smith,+16-280-4352-30,Father,$2000,People Operations,Office Manager,Remote,Remote,Remote,Remote,Remote
O53LUV,Eve,Wilson,US,New York,Madison Avenue,335,John Jones,+31-604-2134-39,Mother,$1800,Support,Customer Support,ESB Office,US,New York,Fifth Avenue,350.0
SUEX6P,David,Johnson,GB,London,Baker Street,105,Bob Brown,+35-753-5910-79,Sister,$3200,Data,Data Analyst,WeWork Office,GB,London,Old Street,207.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3Q85AT,Grace,Davis,US,New York,5th Avenue,180,Carol Wilson,+73-147-3484-43,Friend,$3100,Design,UX Designer,ESB Office,US,New York,Fifth Avenue,350.0
LGT30D,Ivy,Johnson,FR,Paris,Rue de l'Université,414,Frank Brown,+32-656-3523-31,Mother,$3700,Product,Product Manager,Remote,Remote,Remote,Remote,Remote
F7666L,Eve,Johnson,FR,Lyon,Boulevard Haussmann,169,Hank Smith,+4-338-4828-70,Friend,$4500,Leadership,CEO,Remote,Remote,Remote,Remote,Remote


![Employees Final Table](<images/5th Entry/employees_final.png>)

*Please note that, since the company only has offices in London, New York, and Leuven, employees from other countries work remotely.

Again, the code I have uploaded is simple, but it illustrates a common use-case in a data analyst's daily life. If you want to know more about how Python can be used in data analysis, make sure to check the upcoming entries: I'll cover many topics from ingesting the necessary data for our analyses to finally crafting visualizations for presenting our insights to stakeholders.